# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FatimaNdeem/Flyrank-ml-internship./blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
%pip -q install duckdb huggingface_hub

import os
import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except:
    HF_TOKEN = os.environ.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected successfully!")

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name}: {n:,} rows")

Connected successfully!
dim_clients: 104 rows
dim_content: 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily: 78,835,655 rows
fact_query_90d: 2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


- **One row:** One content item for one client on one report date.
- **Tables used:** `fact_content_daily_performance`, with supporting information from `dim_content` and `dim_clients` if needed.
- **Time window:** March 2026 (`month = '2026-03'`).
- **Prediction target (label/proxy):** Whether a content item's search performance is likely to decline, using impressions as the performance signal.
- **Excluded:** June 2026 (the final month) because it is reserved as a holdout/test period and should not be used for developing labels or features.

In [16]:
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


**Features**
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- report_date
- content_hash_id
- client_hash_id

**Label / Proxy**
- Whether a content item's search performance declines (using impressions as the proxy).

**Context**
- report_date
- month
- client_hash_id
- content_hash_id

**Excluded**
- June 2026 data, because it is the final month and should remain an unseen test period.
- Any future information that would not be available at the prediction time, to avoid data leakage.

In [17]:
con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

#### Query 1 – Grain
This query checks whether there are duplicate rows for the same client, content item, and report date. The result is an empty table, confirming that one row represents one client, one content item, and one report date.

#### Query 2 – Row Count and Date Span
This query confirms that the selected data is only from March 2026 and reports the total number of rows.

#### Query 3 – Availability Check
This query filters the data using `gsc_data_available IS TRUE` and counts how many rows remain.

In [18]:
con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


In [19]:
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

,rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [20]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
  AND gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limits

This dataset has several limitations:

- The panel is **unbalanced**, meaning different clients have different amounts of historical data.
- Some clients only have **Google Search Console (GSC)** data, while others also have **GA4** data.
- The data is **pseudonymized**, so client identities, URLs, and search queries cannot be identified.
- The final month (June 2026) should not be used for feature engineering or label development because it serves as the holdout/test period.
- This dataset supports decision-making but cannot explain why search performance changed, since external factors (such as algorithm updates, seasonality, or content edits) are not included.

In [21]:
con.sql(f"""
SELECT
    client_hash_id,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT report_date) AS active_days
FROM {TABLES['fact_daily']}
GROUP BY client_hash_id
ORDER BY active_days ASC
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,first_date,last_date,active_days
0,client_aef6ffea193da149,2026-05-28,2026-06-30,34
1,client_04660893ae39614a,2026-05-22,2026-06-30,40
2,client_a22068e339bf95f5,2026-05-22,2026-06-30,40
3,client_c353557474475e51,2026-04-30,2026-06-30,52
4,client_1a8bf67cad4ee525,2026-05-09,2026-06-30,53
5,client_c7c2962f1c9c3089,2026-05-09,2026-06-30,53
6,client_7de9989c909e91a5,2026-04-20,2026-06-30,72
7,client_9c26c096d6e57253,2026-04-16,2026-06-30,76
8,client_0b245132bb722950,2026-04-06,2026-06-30,86
9,client_8ddc46da5414ffd8,2026-04-02,2026-06-30,90


# 5. Five Feature Frame

The following five features are built from the March 2026 data.

| Feature | Available when? |
|---------|------------------|
| gsc_impressions | Knowable at the decision moment because impressions are already recorded before making a prediction. |
| gsc_clicks | Knowable at the decision moment because clicks are historical observations. |
| gsc_avg_position | Knowable at the decision moment because average position is calculated from previous search performance. |
| client_hash_id | Knowable at the decision moment because the client identity (hashed) is already known. |
| content_hash_id | Knowable at the decision moment because the content item already exists before prediction. |

In [22]:
feature_frame = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
  AND gsc_data_available IS TRUE
LIMIT 10
""").df()

feature_frame
# Remove the leaked feature after the demonstration
honest_feature_frame = leak_demo.drop(columns=["leaked_feature"])
honest_feature_frame.head()

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727


# 6. Leakage Demonstration

To demonstrate data leakage, I intentionally copied gsc_impressions into a new column called leaked_feature. Because this feature contains the same information as the value being predicted, it introduces data leakage and would produce overly optimistic model performance. I then removed the leaked feature and kept only features that would be available before prediction.

In [23]:
leak_demo = feature_frame.copy()

# Deliberately create a label-derived column
leak_demo["leaked_feature"] = leak_demo["gsc_impressions"]

leak_demo.head()

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,leaked_feature
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,20
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,1
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,125
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,7
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,11


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.